# 融合 Softmax

本教程将编写一个融合的 Softmax 算子。对于每一行都能放入 GPU SRAM 的矩阵，教程旨在使其明显快于 PyTorch 原生实现。

你将学习：

- 内核融合对受内存带宽限制的运算有什么好处。
- Triton 中的归约运算。

---

来源：[Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html)，版本为 2026-10-16 下载的官网 main 文档。

中文可运行版：代码依据官方 02 教程补全，保留中文说明。使用当前目录的 uv 环境作为 Notebook 内核，从上到下执行；最后一个单元运行完整 benchmark 并绘图。原文许可见 [TRITON-LICENSE.txt](TRITON-LICENSE.txt)。

## 动机

为逐元素加法编写自定义 GPU 内核很适合入门，但实际应用价值有限。接下来考虑一个经过数值稳定处理的简单 Softmax 运算。

In [ ]:
import torch

import triton
import triton.language as tl
from triton.runtime import driver
from einops import reduce, rearrange

DEVICE = triton.runtime.driver.active.get_active_torch_device()

def naive_softmax(x):
    # x: (M, N)

    # 读 M * N 给元素, 写 M 个元素
    x_max = reduce(x, "m n -> m", "max")
    # 读 M * N + M 个元素, 写 M * N 个元素
    z = x - rearrange(x_max, "m -> m 1")
    # 读 M * N 个元素, 写 M * N 个元素
    numerator = torch.exp(z)
    # 读 M * N 个元素, 写 M 个元素
    denominator = reduce(numerator, "m n -> m", "sum")
    # 读 M * N + M 个元素, 写 M * N 个元素
    ret = numerator / rearrange(denominator, "m -> m 1")
    return ret

对于输入 $x \in \mathbb{R}^{M \times N}$，朴素的 PyTorch 实现需要从 DRAM 读取 $5MN+2M$ 个元素，并写回 $3MN+2M$ 个元素。

这些访存开销很浪费。理想的融合内核只读取一次输入 X，并在芯片内部完成所需计算。这样只需读取 $MN$ 个元素、写回 $MN$ 个元素，理论加速比约为 4 倍，即 $(8MN+4M)/(2MN)$。

原文将这里的 $MN$ 写为“字节”；按前后文的访存计数，应理解为元素数量，实际字节数还要乘以每个元素的大小。

TorchScript 的脚本编译机制试图自动完成这种内核融合，但原教程指出，其效果仍可能不理想。

## 计算内核

每个程序实例以程序实例总数为步长，处理输入矩阵 X 的一组行：加载数据、完成归一化，然后将结果写入输出 Y。

这里需要注意，Triton 的块元素数量要求是 2 的幂。为了支持任意输入形状，需要在内部对各行进行填充，并恰当地保护内存访问。

In [ ]:
@triton.jit
def softmax_kernel(
    output_ptr, input_ptr, input_row_stride, output_row_stride,
    n_rows, n_cols, BLOCK_SIZE: tl.constexpr, num_stages: tl.constexpr
):
    # input_row_stride: 输入移动一行, 需要跨多少个元素
    # output_row_stride: 输出移动一行, 需要跨多少个元素
    # program           处理行号        row_start       row_step
    # 0                 0, 3, 6, 9          0               3
    # 1                 1, 4, 7             1               3
    # 2                 2, 5, 8             2               3

    # softmax_kernel 将整行计算融合在一次 GPU 内核执行里了, 减少了中间结果的内存读写和内核启动
    

    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)

    # 从 row_start 开始, 每次增加 row_step, 直到行号不再小于 n_rows
    # e.g. 1 -> 4 -> 7
    for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
        row_start_ptr = input_ptr + row_idx * input_row_stride
        # 生成当前行内部偏移 [0, 1, 2, ..., 7]
        col_offsets = tl.arange(0, BLOCK_SIZE)
        # 生成一组元素地址
        input_ptrs = row_start_ptr + col_offsets
        # BLOCK_SIZE=8, n_cols = 5
        # [T, T, T, T, T, F, F, F]
        mask = col_offsets < n_cols
        row = tl.load(input_ptrs, mask=mask, other=-float('inf'))
        row_minus_max = row - tl.max(row, axis=0)
        numerator = tl.exp(row_minus_max)
        denominator = tl.sum(numerator, axis=0)
        softmax_output = numerator / denominator
        # 将结果写入 DRAM
        output_row_start_ptr = output_ptr + row_idx * output_row_stride
        output_ptrs = output_row_start_ptr + col_offsets
        tl.store(output_ptrs, softmax_output, mask=mask)

可以创建一个辅助函数，为给定的输入张量启动内核，并传入所需参数和元参数。

2026-10-17 参数调优：保留文档 CUDA 包装函数为 softmax_baseline；softmax 使用每行一个 program、2 或 4 个 warp、1 个流水线阶段。配置针对 RTX 4080 SUPER 的教程 FP32 benchmark，其他硬件和形状未承诺最优。计算内核与 naive 实现保持原样。最后的图同时比较调优前后、PyTorch 和 naive。

In [ ]:
properties = driver.active.utils.get_device_properties(DEVICE.index)
# GPU SM 数量, 有多少个流式多处理器
NUM_SM = properties["multiprocessor_count"]
# 寄存器资源上限
NUM_REGS = properties["max_num_regs"]
# 共享内存资源上限, 单位为字节
SIZE_SMEM = properties["max_shared_mem"]
# 一个 warp 的线程数量, NVIDIA 通常为 32
WARP_SIZE = properties["warpSize"]
target = triton.runtime.driver.active.get_current_target()
kernels = {}

def softmax_baseline(x):
    n_rows, n_cols = x.shape

    # 不小于列数的最小 2 的幂
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    # 每个 program 使用 8 个 warp
    # NVIDIA GPU 上, 一个 warp 32 个线程, 这里每个 program 使用 8 x 32 = 256 个线程
    num_warps = 8

    num_stages = 4 if SIZE_SMEM > 200000 else 2

    y = torch.empty_like(x)

    kernel = softmax_kernel.warmup(
        y, x, x.stride(0), y.stride(0),
        n_rows, n_cols, BLOCK_SIZE=BLOCK_SIZE,
        num_stages=num_stages, num_warps=num_warps, grid=(1, ))

    kernel._init_handles()
    n_regs = kernel.n_regs
    size_smem = kernel.metadata.shared

    occupancy = NUM_REGS // (n_regs * WARP_SIZE * num_warps)
    occupancy = min(occupancy, SIZE_SMEM // size_smem)

    num_programs = NUM_SM * occupancy
    num_programs = min(num_programs, n_rows)

    kernel[(num_programs, 1, 1)](y, x, x.stride(0), y.stride(0), n_rows, n_cols, BLOCK_SIZE, num_stages)

    return y

def softmax(x):
    # RTX 4080 SUPER 上针对教程 FP32 输入实测的简单配置。
    # 每行一个 program，由 GPU 调度；保留上面的文档版本用于对比。
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    num_warps = 2 if BLOCK_SIZE <= 2048 else 4
    num_stages = 1
    y = torch.empty_like(x)
    softmax_kernel[(n_rows,)](
        y, x, x.stride(0), y.stride(0), n_rows, n_cols,
        BLOCK_SIZE=BLOCK_SIZE, num_stages=num_stages, num_warps=num_warps,
    )
    return y


## 单元测试

使用行数和列数都不规则的矩阵测试内核，以检验填充机制是否正确。

In [ ]:
torch.manual_seed(0)
x = torch.randn(1823, 781, device=DEVICE)
y_triton = softmax(x)
y_torch = torch.softmax(x, axis=1)
assert torch.allclose(y_triton, y_torch), (y_triton, y_torch)

检查自定义内核的结果是否与参考实现一致。

## 性能基准测试

固定输入矩阵为 4096 行，改变列数来测试算子的性能，并与 PyTorch 原生 Softmax 以及前面定义的朴素实现进行比较。

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['N'],  # argument names to use as an x-axis for the plot
        x_vals=[128 * i for i in range(2, 100)],  # different possible values for `x_name`
        line_arg='provider',  # argument name whose value corresponds to a different line in the plot
        line_vals=['triton', 'triton_baseline', 'torch', 'naive_softmax'],  # possible values for `line_arg``
        line_names=["Triton Tuned", "Triton Baseline", "Torch", "Naive Softmax"],  # label name for the lines
        styles=[('blue', '-'), ('gray', '--'), ('green', '-'), ('red', '-')],  # line styles
        ylabel="GB/s",  # label name for the y-axis
        plot_name="softmax-performance",  # name for the plot. Used also as a file name for saving the plot.
        args={'M': 4096},  # values for function arguments not in `x_names` and `y_name`
    ))
def benchmark(M, N, provider):
    x = torch.randn(M, N, device=DEVICE, dtype=torch.float32)
    if provider == 'torch':
        ms = triton.testing.do_bench(lambda: torch.softmax(x, axis=-1))
    if provider == 'triton':
        ms = triton.testing.do_bench(lambda: softmax(x))
    if provider == 'triton_baseline':
        ms = triton.testing.do_bench(lambda: softmax_baseline(x))
    if provider == 'naive_softmax':
        ms = triton.testing.do_bench(lambda: naive_softmax(x))
    gbps = lambda ms: 2 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms)


benchmark.run(show_plots=True, print_data=True)

运行上面的 benchmark 后，可以比较当前 GPU 上三种实现的性能。实际结果取决于硬件、输入形状和软件版本，不能直接套用官网的加速比。

PyTorch 的 Softmax 是更通用的操作，支持任意形状的张量；本教程内核按行处理二维张量，并假设列方向连续。
